## # Ironhack Data Analytics Bootcamp
**Author:** Erzsebet Fuzes  
**Cohort:** DAFT OCT2026  
**Topic:** Lab | Hypothesis Testing

# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

In [2]:
import sys
import pandas as pd

# Add the package manager folder of the Anaconda base environment to the Python path,
# so that the notebook can also access the SciPy installed in Anaconda from within the .venv!
anaconda_site_packages = r"C:\Users\fuzes\anaconda3\Lib\site-packages"
if anaconda_site_packages not in sys.path:
    sys.path.insert(0, anaconda_site_packages)

# Import SciPy from Anaconda base!
from scipy.stats import chi2_contingency

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [11]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np

from scipy import stats



In [12]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv"
pokemon = pd.read_csv(url)

pokemon.head()

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [15]:

dragon_hp = pokemon[pokemon['Type 1'] == 'Dragon']['HP']
grass_hp = pokemon[pokemon['Type 1'] == 'Grass']['HP']

print(f"Dragon (Type 1) mean HP: {dragon_hp.mean():.2f} (n={len(dragon_hp)})")
print(f"Grass (Type 1) mean HP:  {grass_hp.mean():.2f} (n={len(grass_hp)})")

t_stat, p_val = stats.ttest_ind(dragon_hp, grass_hp, equal_var=False, alternative='greater')

print(f"\nT-stat: {t_stat:.4f}")
print(f"P-_value: {p_val:.6f}")

if p_val < 0.05:
    print("Conclusion: We reject H0. The average HP of Dragon-type Pokémon is significantly higher.")
else:
    print("Conclusion: We cannot reject H0.")

# --- B) OPCIÓ: Ha a Type 1 VAGY Type 2 is számít (kizárva a duális Dragon/Grass Pokémonokat) ---
is_dragon = (pokemon["Type 1"] == "Dragon") | (pokemon["Type 2"] == "Dragon")
is_grass = (pokemon["Type 1"] == "Grass") | (pokemon["Type 2"] == "Grass")

dragon_all_hp = pokemon[is_dragon & ~is_grass]["HP"]
grass_all_hp = pokemon[is_grass & ~is_dragon]["HP"]

t_stat_all, p_val_all = stats.ttest_ind(dragon_all_hp, grass_all_hp, equal_var=False, alternative="greater")
print(f"\n[Type 1 vagy 2] Dragon mean: {dragon_all_hp.mean():.2f}, Grass átlag: {grass_all_hp.mean():.2f}, p-érték: {p_val_all:.6f}")

Dragon (Type 1) mean HP: 83.31 (n=32)
Grass (Type 1) mean HP:  67.27 (n=70)

T-stat: 3.3350
P-_value: 0.000799
Conclusion: We reject H0. The average HP of Dragon-type Pokemon is significantly higher.

[Type 1 vagy 2] Dragon mean: 83.16, Grass átlag: 66.01, p-érték: 0.000051


- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [16]:
import pandas as pd
from scipy import stats

stats_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

legendary = pokemon[pokemon['Legendary'] == True]
non_legendary = pokemon[pokemon['Legendary'] == False]

results = []

for col in stats_cols:
    t_stat, p_val = stats.ttest_ind(legendary[col], non_legendary[col], equal_var=False, alternative='two-sided')
    
    results.append({
        'Stat': col,
        'Legendary Mean': round(legendary[col].mean(), 2),
        'Non-Legendary Mean': round(non_legendary[col].mean(), 2),
        'T-stat': round(t_stat, 4),
        'P-value': p_val,
        'Reject H0 (p < 0.05)': p_val < 0.05
    })

results_df = pd.DataFrame(results)
display(results_df)

,Stat,Legendary Mean,Non-Legendary Mean,T-stat,P-value,Reject H0 (p < 0.05)
0,HP,92.74,67.18,8.9814,1.002691e-13,True
1,Attack,116.68,75.67,10.4381,2.520372e-16,True
2,Defense,99.66,71.56,7.6371,4.826998e-11,True
3,Sp. Atk,122.18,68.45,13.4174,1.551461e-21,True
4,Sp. Def,105.94,68.89,10.0157,2.294933e-15,True
5,Speed,100.18,65.46,11.4750,1.049016e-18,True


**Conclusion**
For all six battle statistics (HP, Attack, Defense, Sp. Atk, Sp. Def, and Speed), the p-values are well below the 5% significance level (alpha = 0.05). Therefore, we reject the null hypothesis (H_0) across all attributes and conclude that Legendary Pokémon have statistically significantly different (specifically, much higher) average stats compared to Non-Legendary Pokémon.

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [18]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv"
housing = pd.read_csv(url)

housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [20]:
# defining a function for calculating euclidean distance
def euclidean_distance(lon1, lat1, lon2, lat2):
    return np.sqrt((lon1 - lon2)**2 + (lat1 - lat2)**2)

# 1. Coordinates
school_coords = (-118, 34)
hospital_coords = (-122, 37)

# 2. Calculating distances to every house (neighborhood)
housing["dist_school"] = euclidean_distance(housing["longitude"], housing["latitude"], school_coords[0], school_coords[1])
housing["dist_hospital"] = euclidean_distance(housing["longitude"], housing["latitude"], hospital_coords[0], hospital_coords[1])

# 3. Grouping: close to EITHER the school OR the hospital (distance < 0.50)
is_close = (housing["dist_school"] < 0.50) | (housing["dist_hospital"] < 0.50)

close_houses = housing[is_close]["median_house_value"]
far_houses = housing[~is_close]["median_house_value"]

print(f"Average price of nearby houses: ${close_houses.mean():,.2f} (n={len(close_houses)})")
print(f"Average price of remote houses: ${far_houses.mean():,.2f} (n={len(far_houses)})")

# 4. One-sided independent two-sample Welch’s t-test (alternative='greater', because we are testing whether close > far)
t_stat, p_val = stats.ttest_ind(close_houses, far_houses, equal_var=False, alternative="greater")

print(f"\nT-stat: {t_stat:.4f}")
print(f"P-value:       {p_val:.6e}")

if p_val < 0.05:
    print("Conclusion: We reject H0. The average price of nearby houses is significantly higher.")
else:
    print("Conclusion: We cannot reject H0.")

Average price of nearby houses: $246,951.98 (n=6829)
Average price of remote houses: $180,678.44 (n=10171)

T-stat: 37.9923
P-value:       1.503248e-301
Conclusion: We reject H0. The average price of nearby houses is significantly higher.
